In [5]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

load_dotenv

key_file_path = os.environ.get("GCP_KEY_PATH")

client = bigquery.Client.from_service_account_json(
    key_file_path,
    project="quantum-echo-data-eng-prod"
)


In [6]:
df_sales = client.query("SELECT * FROM `quantum-echo-data-eng-prod.gold.fct_sales` WHERE order_date >= '2010-01-01'").to_dataframe()

df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

sales_performance = (
    df_sales.assign(order_year=df_sales['order_date'].dt.strftime("%Y"))
    .groupby("order_year", as_index=False)
    .agg(
        total_sales=('gross_sales_amount', "sum"),
        total_customers=('customer_key', "nunique")
    )
    .assign(
        total_sales=lambda x: x['total_sales'].astype(int),
        total_customers=lambda x: x['total_customers'].astype(int),
        running_total_sales=lambda x: x["total_sales"].cumsum(),
        running_total_customers=lambda x: x["total_customers"].cumsum()
    )
)

sales_performance['total_sales'] = sales_performance['total_sales'].astype('int64')
sales_performance

,order_year,total_sales,total_customers,running_total_sales,running_total_customers
0,2010,43419,14,43419,14
1,2011,7075088,2216,7118507,2230
2,2012,5842231,3255,12960738,5485
3,2013,16344494,17427,29305232,22912
4,2014,45642,834,29350874,23746
